In [32]:
from gftmx.gfpmx_data import gfpmx_data
import numpy as np
import pandas

# Introduction

The purpose of this document is to describe an implementation of the Global Forest Products Trade Model in a cobweb form.
It is inspired by Joseph Buongiorno's cobweb GFPMX: "A Cobweb Model of the Global Forest Sector, 
with an Application to the Impact of the COVID-19 Pandemic" https://doi.org/10.3390/su13105507. 
The GFPMX input data and parameters are available as a spreadsheet at: https://buongiorno.russell.wisc.edu/gfpm/. Our first goal is to reproduce those results using the same input data. Our second goal is to update the input data from FAOSTAT and perform the computation.

# Data structure

The variables production $P$, import $I$, export $E$ and demand $D$ are distributed accross commodity $c$, reporter $r$, and time $t$. In each country $r$, the variables are related through the following equation accross all values of $c$, and $t$:

$$P_{crt} + I_{crt} - E_{crt} = D_{crt}$$ 

# Sample data from GFTMX

## Sawnwood

In [33]:
swd_sheets = gfpmx_data.list_sheets().query("product=='sawn'")
known_columns = ['id', 'year', 'unnamed_2', 'constant', 'unnamed_1', 'faostat_name', 'country', 'value']
print("Additional variables besides the value in each sheet:")
for s in swd_sheets.name:
    print("  ", s, set(gfpmx_data[s].columns) - set(known_columns))
    #display(gfpmx_data[s].head(1))

Additional variables besides the value in each sheet:
   sawncons {'price_elasticity', 'gdp_elasticity'}
   sawncons_usd set()
   sawnconspercap set()
   sawnexp {'marginal_propensity_to_export'}
   sawnexp_usd set()
   sawnimp {'price_elasticity', 'gdp_elasticity'}
   sawnimp_usd {'price_elasticity', 'gdp_elasticity'}
   sawnnettrade set()
   sawnnettrade_usd set()
   sawnprice {'world_price_elasticity', 'unnamed_4'}
   sawnprod set()
   sawnprod_usd set()
   sawntariff set()


In [8]:
swd_cons = gfpmx_data['sawncons']
print(swd_cons.columns)
print(swd_cons.year.unique())
index = ["faostat_name", "country", "year"]
#index = ["country", "year"]
swd_cons = swd_cons.set_index(index)
swd_cons

Index(['id', 'year', 'unnamed_2', 'constant', 'unnamed_1', 'gdp_elasticity',
       'faostat_name', 'country', 'price_elasticity', 'value'],
      dtype='object')
[1992 1993 1994 1995 1996 1997 1998 1999 2000 2001 2002 2003 2004 2005
 2006 2007 2008 2009 2010 2011 2012 2013 2014 2015 2016 2017 2018 2019
 2020 2021 2022 2023 2024 2025 2026 2027 2028 2029 2030 2031 2032 2033
 2034 2035 2036 2037 2038 2039 2040 2041 2042 2043 2044 2045 2046 2047
 2048 2049 2050 2051 2052 2053 2054 2055 2056 2057 2058 2059 2060 2061
 2062 2063 2064 2065 2066 2067 2068 2069 2070]


id unnamed_2  constant    unnamed_1  \
faostat_name      country       year                                         
Sawnwood+sleepers Algeria       1992    0    1000m3  0.006851  Consumption   
                  Angola        1992    1    1000m3  3.653224  Consumption   
                  Benin         1992    2    1000m3  0.001572  Consumption   
                  Botswana      1992    3    1000m3  0.000007  Consumption   
                  Burkina Faso  1992    4    1000m3  0.000715  Consumption   
...                                   ...       ...       ...          ...   
                  NORTH AMERICA 2070  182    1000m3       NaN  Consumption   
                  SOUTH AMERICA 2070  183    1000m3       NaN  Consumption   
                  ASIA          2070  184    1000m3       NaN  Consumption   
                  OCEANIA       2070  185    1000m3       NaN  Consumption   
                  EUROPE        2070  186    1000m3       NaN  Consumption   

                                      gdp_elasticity  price_elasticity  \
faostat_name      country       year                                     
Sawnwood+sleepers Algeria       1992        0.852714         -0.636577   
                  Angola        1992        0.054637         -0.056622   
                  Benin         1992        1.000000         -0.781967   
                  Botswana      1992        1.000000         -0.161713   
                  Burkina Faso  1992        0.802507         -0.601820   
...                                              ...               ...   
                  NORTH AMERICA 2070             NaN               NaN   
                  SOUTH AMERICA 2070             NaN               NaN   
                  ASIA          2070             NaN               NaN   
                  OCEANIA       2070             NaN               NaN   
                  EUROPE        2070             NaN               NaN   

                                              value  
faostat_name      country       year                 
Sawnwood+sleepers Algeria       1992     277.000000  
                  Angola        1992       8.000000  
                  Benin         1992      31.000000  
                  Botswana      1992       0.000000  
                  Burkina Faso  1992       2.000000  
...                                             ...  
                  NORTH AMERICA 2070  129114.114413  
                  SOUTH AMERICA 2070   34844.412139  
                  ASIA          2070  392739.412837  
                  OCEANIA       2070   11276.256263  
                  EUROPE        2070  137330.412006  

[14773 rows x 7 columns]

In [5]:
#help(swd_cons.index)

In [6]:
swd_cons.loc[("Sawnwood+sleepers", "Algeria", 1992)]

id                            0
price_elasticity      -0.636577
unnamed_1           Consumption
gdp_elasticity         0.852714
constant               0.006851
unnamed_2                1000m3
value                     277.0
Name: (Sawnwood+sleepers, Algeria, 1992), dtype: object

In [20]:
swd_cons.loc[("Sawnwood+sleepers", "Algeria", 2017)]

id                            0
unnamed_2                1000m3
constant               0.006851
unnamed_1           Consumption
gdp_elasticity         0.852714
price_elasticity      -0.636577
value                    2213.0
Name: (Sawnwood+sleepers, Algeria, 2017), dtype: object

##  Other products

In [26]:
sheets = gfpmx_data.list_sheets()
for prod in sheets["product"].unique():
    sheets_selected = sheets.query("product==@prod")
    known_columns = ['id', 'year', 'unnamed_2', 'constant', 'unnamed_1', 'faostat_name', 'country', 'value']
    print(f"Additional variables in {prod}-related sheets:")
    for s in sheets_selected.name:
        print("  ", s, set(gfpmx_data[s].columns) - set(known_columns))
        #display(gfpmx_data[s].head(1))

Additional variables in fuel-related sheets:
   fuelcons {'price_elasticity', 'gdp_elasticity'}
   fuelcons_usd {'price_elasticity', 'gdp_elasticity'}
   fuelconspercap set()
   fuelexp {'marginal_propensity_to_export'}
   fuelexp_usd {'marginal_propensity_to_export'}
   fuelimp {'price_elasticity', 'gdp_elasticity'}
   fuelimp_usd set()
   fuelnettrade set()
   fuelnettrade_usd set()
   fuelprice {'world_price_elasticity', 'unnamed_4'}
   fuelprod {'element', 'unit'}
   fuelprod_usd set()
   fueltariff set()
Additional variables in indround-related sheets:
   indroundcons {'products_elasticity', 'price_elasticity'}
   indroundcons_usd set()
   indroundexp {'unnamed_6', 'unnamed_5'}
   indroundexp_usd set()
   indroundimp {'products_elasticity', 'price_elasticity'}
   indroundimp_usd set()
   indroundnettrade set()
   indroundnettrade_usd set()
   indroundprice {'unnamed_3', 'world_price_elasticity', 'unnamed_4'}
   indroundprod {'element', 'unit'}
   indroundprod_usd set()
   indround

# Other Variables

In [31]:
sheets_selected = sheets.query("product!=product")
known_columns = ['id', 'year', 'unnamed_2', 'constant', 'unnamed_1', 'faostat_name', 'country', 'value']
print(f"Additional variables in {prod}-related sheets:")
for s in sheets_selected.name:
    print("  ", s, set(gfpmx_data[s].columns) - set(known_columns))
    #display(gfpmx_data[s].head(1))

Additional variables in nan-related sheets:
   area {'linear_effect_of_gdp_cap', 'growth_rate_in_2018', 'exponential_effect_of_gdp_cap', 'unnamed_0'}
   author {'gfpmx_a_cobweb_model_of_the_global_forest_sector'}
   gdp {'unnamed_0'}
   gdppercap {'unnamed_0'}
   harvestperha {'unnamed_0'}
   harvestperstock {'unnamed_0'}
   names {'faostat', 'faostat_1', 'unnamed_10', 'gfpm_x', 'unnamed_7', 'gfpm_x_1'}
   notes {'updated'}
   population {'unnamed_0'}
   stock {'stock_growth_rate_without_harvest', 'harvest_effect_on_stock', 'unnamed_0'}
   stockpercap {'unnamed_0'}
   stockperha {'unnamed_0'}
   totalcons_usd set()
   totalexp_usd set()
   totalimp_usd set()
   totalnettrade_usd set()
   totalprod_usd set()
   valueadded set()


EmptyDataError: No columns to parse from file

# np.array test

In [73]:
X_test= np.array([[0.33279229, 0.52539847, 0.32301045, 0.08504233],
[0.18115765, 0.11468587, 0.34419565, 0.91663379],
[0.55340608, 0.80923817, 0.24275124, 0.83075826],
[0.96856274, 0.43972433, 0.02564373, 0.51602777],
[0.5127286 , 0.88340132, 0.5584916 , 0.36889873],
[0.29306659, 0.82146901, 0.48855817, 0.93730368],
[0.9914627 , 0.81634472, 0.58954755, 0.9969822 ],
[0.3480635 , 0.59528288, 0.96056427, 0.41617849],
[0.88517647, 0.36977957, 0.69322023, 0.09770496],
[0.9115624 , 0.50818111, 0.94525091, 0.53483017],
[0.8379305 , 0.91641027, 0.03112315, 0.55125694],
[0.02452024, 0.72898607, 0.28460244, 0.34467602],
[0.17242202, 0.05892018, 0.19770165, 0.37108781],
[0.01634621, 0.93051447, 0.45819514, 0.71471246],
[0.84937553, 0.23620682, 0.60876167, 0.30354231]])
display(X_test)
X_test @ np.array(range(4))

array([[0.33279229, 0.52539847, 0.32301045, 0.08504233],
       [0.18115765, 0.11468587, 0.34419565, 0.91663379],
       [0.55340608, 0.80923817, 0.24275124, 0.83075826],
       [0.96856274, 0.43972433, 0.02564373, 0.51602777],
       [0.5127286 , 0.88340132, 0.5584916 , 0.36889873],
       [0.29306659, 0.82146901, 0.48855817, 0.93730368],
       [0.9914627 , 0.81634472, 0.58954755, 0.9969822 ],
       [0.3480635 , 0.59528288, 0.96056427, 0.41617849],
       [0.88517647, 0.36977957, 0.69322023, 0.09770496],
       [0.9115624 , 0.50818111, 0.94525091, 0.53483017],
       [0.8379305 , 0.91641027, 0.03112315, 0.55125694],
       [0.02452024, 0.72898607, 0.28460244, 0.34467602],
       [0.17242202, 0.05892018, 0.19770165, 0.37108781],
       [0.01634621, 0.93051447, 0.45819514, 0.71471246],
       [0.84937553, 0.23620682, 0.60876167, 0.30354231]])

array([1.42654636, 3.55297854, 3.78701543, 2.0390951 , 3.10708071,
       4.61049639, 4.98638642, 3.76494689, 2.04933491, 4.00317344,
       2.63242739, 2.33221901, 1.56758691, 3.99104213, 2.36435709])

In [68]:
X = np.array([[0.33279229, 0.52539847, 0.32301045, 0.08504233],
              [0.18115765, 0.11468587, 0.34419565, 0.91663379],
              [0.55340608, 0.80923817, 0.24275124, 0.83075826],
              [0.96856274, 0.43972433, 0.02564373, 0.51602777]])



In [70]:
# TODO: Check how to convert a data frame indexed column into a multi dimentional array
help(np.array)

Help on built-in function array in module numpy:

array(...)
    array(object, dtype=None, *, copy=True, order='K', subok=False, ndmin=0,
          like=None)
    
    Create an array.
    
    Parameters
    ----------
    object : array_like
        An array, any object exposing the array interface, an object whose
        __array__ method returns an array, or any (nested) sequence.
    dtype : data-type, optional
        The desired data-type for the array.  If not given, then the type will
        be determined as the minimum type required to hold the objects in the
        sequence.
    copy : bool, optional
        If true (default), then the object is copied.  Otherwise, a copy will
        only be made if __array__ returns a copy, if obj is a nested sequence,
        or if a copy is needed to satisfy any of the other requirements
        (`dtype`, `order`, etc.).
    order : {'K', 'A', 'C', 'F'}, optional
        Specify the memory layout of the array. If object is not an array